In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Celebal Week 6") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.3


In [5]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd

In [3]:
import random

categories = {
    "Electronics": [
        ("Laptop", 55000),
        ("Phone", 25000),
        ("Tablet", 18000),
        ("Monitor", 12000),
        ("Keyboard", 1500),
        ("Mouse", 800),
        ("Headphones", 3000),
        ("Camera", 45000)
    ],
    "Furniture": [
        ("Chair", 2500),
        ("Table", 6000),
        ("Desk", 7000),
        ("Sofa", 18000),
        ("Bed", 30000)
    ],
    "Grocery": [
        ("Rice Bag", 1200),
        ("Cooking Oil", 1800),
        ("Sugar", 700),
        ("Tea Pack", 500),
        ("Coffee", 900)
    ]
}

statuses = ["Completed", "Pending", "Cancelled"]
regions = ["North", "South", "East", "West"]
priorities = ["High", "Medium", "Low"]

data = []

for i in range(1, 41):

    category = random.choice(list(categories.keys()))
    product_name, price = random.choice(categories[category])

    base_price = round(price / 1.18, 2)
    amount = price

    user_id = None if random.random() < 0.15 else f"U{100+i}"

    data.append((
        1000 + i,
        f"P{100+i}",
        product_name,
        category,
        str(price),
        base_price,
        amount,
        random.choice(statuses),
        random.choice(regions),
        random.choice(priorities),
        user_id,
        f"Item {i}"
    ))

columns = [
    "order_id",
    "product_id",
    "product_name",
    "category",
    "price",
    "base_price",
    "amount",
    "status",
    "region",
    "priority",
    "user_id",
    "old_name"
]

In [4]:
df = spark.createDataFrame(data, columns)

df.show(10, truncate=False)
df.printSchema()

+--------+----------+------------+-----------+-----+----------+------+---------+------+--------+-------+--------+
|order_id|product_id|product_name|category   |price|base_price|amount|status   |region|priority|user_id|old_name|
+--------+----------+------------+-----------+-----+----------+------+---------+------+--------+-------+--------+
|1001    |P101      |Cooking Oil |Grocery    |1800 |1525.42   |1800  |Pending  |South |High    |U101   |Item 1  |
|1002    |P102      |Tea Pack    |Grocery    |500  |423.73    |500   |Completed|South |Low     |U102   |Item 2  |
|1003    |P103      |Tablet      |Electronics|18000|15254.24  |18000 |Cancelled|South |Medium  |U103   |Item 3  |
|1004    |P104      |Camera      |Electronics|45000|38135.59  |45000 |Pending  |North |Medium  |U104   |Item 4  |
|1005    |P105      |Table       |Furniture  |6000 |5084.75   |6000  |Completed|North |High    |U105   |Item 5  |
|1006    |P106      |Sofa        |Furniture  |18000|15254.24  |18000 |Pending  |East  |M

In [6]:
df.write \
.mode("overwrite") \
.option("header", True) \
.csv("data/source")

In [7]:
df_csv = spark.read.csv(
    "data/source",
    header=True,
    inferSchema=True
)

df_csv.show(5)
df_csv.printSchema()

+--------+----------+------------+-----------+-----+----------+------+---------+------+--------+-------+--------+
|order_id|product_id|product_name|   category|price|base_price|amount|   status|region|priority|user_id|old_name|
+--------+----------+------------+-----------+-----+----------+------+---------+------+--------+-------+--------+
|    1001|      P101| Cooking Oil|    Grocery| 1800|   1525.42|  1800|  Pending| South|    High|   U101|  Item 1|
|    1002|      P102|    Tea Pack|    Grocery|  500|    423.73|   500|Completed| South|     Low|   U102|  Item 2|
|    1003|      P103|      Tablet|Electronics|18000|  15254.24| 18000|Cancelled| South|  Medium|   U103|  Item 3|
|    1004|      P104|      Camera|Electronics|45000|  38135.59| 45000|  Pending| North|  Medium|   U104|  Item 4|
|    1005|      P105|       Table|  Furniture| 6000|   5084.75|  6000|Completed| North|    High|   U105|  Item 5|
+--------+----------+------------+-----------+-----+----------+------+---------+------+-

In [8]:
df_csv.write \
.mode("overwrite") \
.parquet("data/parquet")

In [9]:
df_parquet = spark.read.parquet("data/parquet")

df_parquet.show(5)
df_parquet.printSchema()

+--------+----------+------------+-----------+-----+----------+------+---------+------+--------+-------+--------+
|order_id|product_id|product_name|   category|price|base_price|amount|   status|region|priority|user_id|old_name|
+--------+----------+------------+-----------+-----+----------+------+---------+------+--------+-------+--------+
|    1001|      P101| Cooking Oil|    Grocery| 1800|   1525.42|  1800|  Pending| South|    High|   U101|  Item 1|
|    1002|      P102|    Tea Pack|    Grocery|  500|    423.73|   500|Completed| South|     Low|   U102|  Item 2|
|    1003|      P103|      Tablet|Electronics|18000|  15254.24| 18000|Cancelled| South|  Medium|   U103|  Item 3|
|    1004|      P104|      Camera|Electronics|45000|  38135.59| 45000|  Pending| North|  Medium|   U104|  Item 4|
|    1005|      P105|       Table|  Furniture| 6000|   5084.75|  6000|Completed| North|    High|   U105|  Item 5|
+--------+----------+------------+-----------+-----+----------+------+---------+------+-

In [10]:
from pyspark.sql.types import DoubleType

df_modified = df_parquet \
    .withColumnRenamed("old_name", "new_name") \
    .withColumn("price", col("price").cast(DoubleType()))

df_modified.printSchema()
df_modified.show(5)

root
 |-- order_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- amount: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- new_name: string (nullable = true)

+--------+----------+------------+-----------+-------+----------+------+---------+------+--------+-------+--------+
|order_id|product_id|product_name|   category|  price|base_price|amount|   status|region|priority|user_id|new_name|
+--------+----------+------------+-----------+-------+----------+------+---------+------+--------+-------+--------+
|    1001|      P101| Cooking Oil|    Grocery| 1800.0|   1525.42|  1800|  Pending| South|    High|   U101|  Item 1|
|    1002|      P102|    Tea Pack|    Grocery|  50

In [11]:
electronics_df = df_modified.filter(
    col("category") == "Electronics"
).select("product_id", "price")

electronics_df.show()

+----------+-------+
|product_id|  price|
+----------+-------+
|      P103|18000.0|
|      P104|45000.0|
|      P111| 1500.0|
|      P115|18000.0|
|      P119|12000.0|
|      P120|55000.0|
|      P121| 3000.0|
|      P122|25000.0|
|      P125|  800.0|
|      P127|25000.0|
|      P128|12000.0|
|      P132|25000.0|
|      P133| 1500.0|
|      P137|55000.0|
+----------+-------+



In [12]:
completed_orders = df_modified.filter(
    (col("status") == "Completed") &
    (col("amount") > 1000)
)

completed_orders.show()

+--------+----------+------------+-----------+-------+----------+------+---------+------+--------+-------+--------+
|order_id|product_id|product_name|   category|  price|base_price|amount|   status|region|priority|user_id|new_name|
+--------+----------+------------+-----------+-------+----------+------+---------+------+--------+-------+--------+
|    1005|      P105|       Table|  Furniture| 6000.0|   5084.75|  6000|Completed| North|    High|   U105|  Item 5|
|    1008|      P108|        Sofa|  Furniture|18000.0|  15254.24| 18000|Completed| South|    High|   U108|  Item 8|
|    1010|      P110|    Rice Bag|    Grocery| 1200.0|   1016.95|  1200|Completed| North|    High|   U110| Item 10|
|    1011|      P111|    Keyboard|Electronics| 1500.0|   1271.19|  1500|Completed| South|    High|   U111| Item 11|
|    1016|      P116|       Table|  Furniture| 6000.0|   5084.75|  6000|Completed| South|    High|   U116| Item 16|
|    1018|      P118|        Desk|  Furniture| 7000.0|    5932.2|  7000|

In [13]:
df_final = df_modified.withColumn(
    "final_price",
    col("base_price") * 1.18
)

df_final.show(5)

+--------+----------+------------+-----------+-------+----------+------+---------+------+--------+-------+--------+------------------+
|order_id|product_id|product_name|   category|  price|base_price|amount|   status|region|priority|user_id|new_name|       final_price|
+--------+----------+------------+-----------+-------+----------+------+---------+------+--------+-------+--------+------------------+
|    1001|      P101| Cooking Oil|    Grocery| 1800.0|   1525.42|  1800|  Pending| South|    High|   U101|  Item 1|         1799.9956|
|    1002|      P102|    Tea Pack|    Grocery|  500.0|    423.73|   500|Completed| South|     Low|   U102|  Item 2|          500.0014|
|    1003|      P103|      Tablet|Electronics|18000.0|  15254.24| 18000|Cancelled| South|  Medium|   U103|  Item 3|        18000.0032|
|    1004|      P104|      Camera|Electronics|45000.0|  38135.59| 45000|  Pending| North|  Medium|   U104|  Item 4|44999.996199999994|
|    1005|      P105|       Table|  Furniture| 6000.0| 

In [14]:
north_or_high = df_final.filter(
    (col("region") == "North") |
    (col("priority") == "High")
)

north_or_high.show()

+--------+----------+------------+-----------+-------+----------+------+---------+------+--------+-------+--------+------------------+
|order_id|product_id|product_name|   category|  price|base_price|amount|   status|region|priority|user_id|new_name|       final_price|
+--------+----------+------------+-----------+-------+----------+------+---------+------+--------+-------+--------+------------------+
|    1001|      P101| Cooking Oil|    Grocery| 1800.0|   1525.42|  1800|  Pending| South|    High|   U101|  Item 1|         1799.9956|
|    1004|      P104|      Camera|Electronics|45000.0|  38135.59| 45000|  Pending| North|  Medium|   U104|  Item 4|44999.996199999994|
|    1005|      P105|       Table|  Furniture| 6000.0|   5084.75|  6000|Completed| North|    High|   U105|  Item 5|          6000.005|
|    1007|      P107|       Table|  Furniture| 6000.0|   5084.75|  6000|  Pending| North|     Low|   U107|  Item 7|          6000.005|
|    1008|      P108|        Sofa|  Furniture|18000.0| 

In [15]:
clean_df = df_final.filter(col("user_id").isNotNull())

clean_df.show()

+--------+----------+------------+-----------+-------+----------+------+---------+------+--------+-------+--------+------------------+
|order_id|product_id|product_name|   category|  price|base_price|amount|   status|region|priority|user_id|new_name|       final_price|
+--------+----------+------------+-----------+-------+----------+------+---------+------+--------+-------+--------+------------------+
|    1001|      P101| Cooking Oil|    Grocery| 1800.0|   1525.42|  1800|  Pending| South|    High|   U101|  Item 1|         1799.9956|
|    1002|      P102|    Tea Pack|    Grocery|  500.0|    423.73|   500|Completed| South|     Low|   U102|  Item 2|          500.0014|
|    1003|      P103|      Tablet|Electronics|18000.0|  15254.24| 18000|Cancelled| South|  Medium|   U103|  Item 3|        18000.0032|
|    1004|      P104|      Camera|Electronics|45000.0|  38135.59| 45000|  Pending| North|  Medium|   U104|  Item 4|44999.996199999994|
|    1005|      P105|       Table|  Furniture| 6000.0| 

In [16]:
clean_df.write \
.mode("overwrite") \
.option("header", True) \
.csv("output/clean_orders")

In [17]:
print("Total Records:", clean_df.count())

clean_df.select("product_name", "price").show(5)

clean_df.describe().show()

Total Records: 35
+------------+-------+
|product_name|  price|
+------------+-------+
| Cooking Oil| 1800.0|
|    Tea Pack|  500.0|
|      Tablet|18000.0|
|      Camera|45000.0|
|       Table| 6000.0|
+------------+-------+
only showing top 5 rows
+-------+------------------+----------+------------+-----------+-----------------+------------------+-----------------+---------+------+--------+-------+--------+------------------+
|summary|          order_id|product_id|product_name|   category|            price|        base_price|           amount|   status|region|priority|user_id|new_name|       final_price|
+-------+------------------+----------+------------+-----------+-----------------+------------------+-----------------+---------+------+--------+-------+--------+------------------+
|  count|                35|        35|          35|         35|               35|                35|               35|       35|    35|      35|     35|      35|                35|
|   mean|1019.028571428

In [18]:
clean_df.printSchema()
clean_df.show(10, truncate=False)

root
 |-- order_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- amount: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- new_name: string (nullable = true)
 |-- final_price: double (nullable = true)

+--------+----------+------------+-----------+-------+----------+------+---------+------+--------+-------+--------+------------------+
|order_id|product_id|product_name|category   |price  |base_price|amount|status   |region|priority|user_id|new_name|final_price       |
+--------+----------+------------+-----------+-------+----------+------+---------+------+--------+-------+--------+------------------+
|1001    |P101      |Cooking Oil |Grocery    |1800.0 |1525.42   |1